<a href="https://colab.research.google.com/github/GustavoNachbar/churn-dataset-clusters-classify-tests/blob/main/naive_bayes_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

variaveis_numericas = [
    'PontuacaoCredito', 'Idade', 'TempoRelacionamento',
    'Saldo', 'NumeroProdutos', 'SalarioEstimado'
]

# 70% treino, 30% temporário
df_treino, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Exited"]
)

# 15% teste, 15% validação
df_teste, df_validacao = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["Exited"]
)

# Cluster treinado SÓ com cancelados do df_treino (nunca olha teste/validação)
df_cancelados_treino = df_treino[df_treino["Exited"] == 1][variaveis_numericas]

scaler_perfil = StandardScaler()
X_scaled_cancelados = scaler_perfil.fit_transform(df_cancelados_treino)

kmeans_k4 = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_k4.fit(X_scaled_cancelados)

# Aplicado (predict) nas 3 bases separadamente, sem re-treinar em nenhuma delas
for dataframe in [df_treino, df_teste, df_validacao]:
    X_scaled = scaler_perfil.transform(dataframe[variaveis_numericas])
    dataframe['Cluster_k4'] = kmeans_k4.predict(X_scaled)
    dataframe['Distancia_Centroide_k4'] = kmeans_k4.transform(X_scaled).min(axis=1)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

features_numericas = variaveis_numericas + ['Distancia_Centroide_k4']

cluster_dummies_treino = pd.get_dummies(df_treino['Cluster_k4'], prefix='cluster_k4')
cluster_dummies_teste = pd.get_dummies(df_teste['Cluster_k4'], prefix='cluster_k4')
cluster_dummies_treino, cluster_dummies_teste = cluster_dummies_treino.align(
    cluster_dummies_teste, join='outer', axis=1, fill_value=0
)

X_train = pd.concat([df_treino[features_numericas].reset_index(drop=True),
                      cluster_dummies_treino.reset_index(drop=True)], axis=1)
X_test = pd.concat([df_teste[features_numericas].reset_index(drop=True),
                     cluster_dummies_teste.reset_index(drop=True)], axis=1)
y_train = df_treino['Exited']
y_test = df_teste['Exited']

pipeline_nb = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

param_grid_nb = {
    'nb__var_smoothing': np.logspace(0, -9, num=10)
}

grid_search_nb = GridSearchCV(
    estimator=pipeline_nb, param_grid=param_grid_nb, cv=5, scoring='f1', n_jobs=-1
)
grid_search_nb.fit(X_train, y_train)

melhor_modelo = grid_search_nb.best_estimator_
y_pred = melhor_modelo.predict(X_test)
y_proba = melhor_modelo.predict_proba(X_test)[:, 1]

resultado_nb_k4 = {
    'melhores_parametros': grid_search_nb.best_params_,
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
}
pd.DataFrame([resultado_nb_k4])